# WF2020 Recovery Checkpoint Failure Attribution v1

Purpose: explain why `recovery_participation_three_fold_checkpoint_metric_replay_v1` regressed on `wf_2020` before any further checkpoint replay or shared BC-label code change.

Scope:
- compare the latest recovery replay against the prior bull-reentry replay and the cap10 checkpoint replay reference when available;
- audit whether `recovery_participation_target_position` is actually consumed by the current state-objective BC target chooser;
- write sidecar CSV/JSON evidence and a single-run decision.


In [1]:
from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'src/soxl_quant').exists() and (candidate / 'runtime').exists():
            return candidate
    raise FileNotFoundError('could not locate repo root containing src/soxl_quant and runtime')

ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(ROOT / 'src'))

from soxl_quant.policy.policy_builder import state_objective_target_position_from_row

CARD = ROOT / 'runtime/model_selection/card/wf2020_recovery_checkpoint_failure_attribution_v1'
CARD.mkdir(parents=True, exist_ok=True)
RECOVERY_REPLAY = ROOT / 'runtime/model_selection/card/recovery_participation_three_fold_checkpoint_metric_replay_v1'
BULL_REPLAY = ROOT / 'runtime/model_selection/card/bull_reentry_three_fold_checkpoint_metric_replay_v1'
CAP10_REPLAY = ROOT / 'runtime/model_selection/card/cap10_three_fold_checkpoint_metric_replay_v2'
RECOVERY_TARGET = ROOT / 'runtime/model_selection/card/wf2020_recovery_participation_target_v1/recovery_participation_three_fold_target_contract.csv'

for required_path in [RECOVERY_REPLAY / 'checkpoint_candidate_metrics.csv', BULL_REPLAY / 'checkpoint_candidate_metrics.csv', RECOVERY_TARGET]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print('repo root', ROOT)
print('card', CARD.relative_to(ROOT))
print('recovery replay', RECOVERY_REPLAY.relative_to(ROOT))
print('bull replay', BULL_REPLAY.relative_to(ROOT))
print('cap10 replay exists', (CAP10_REPLAY / 'checkpoint_candidate_metrics.csv').exists())


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


repo root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card runtime/model_selection/card/wf2020_recovery_checkpoint_failure_attribution_v1
recovery replay runtime/model_selection/card/recovery_participation_three_fold_checkpoint_metric_replay_v1
bull replay runtime/model_selection/card/bull_reentry_three_fold_checkpoint_metric_replay_v1
cap10 replay exists True


## Replay Metric Delta

The latest recovery replay must be compared to the immediately preceding bull-reentry replay because the recovery contract was meant to repair the `wf_2020` gap without giving up the two folds that already passed. The strict promotion gate still requires all walk-forward folds to pass and MaxDD under 20%.


In [2]:
def load_candidate_metrics(path: Path, label: str) -> pd.DataFrame:
    df = pd.read_csv(path / 'checkpoint_candidate_metrics.csv')
    df = df.copy()
    df['source'] = label
    df['iter'] = pd.to_numeric(df['iter'], errors='coerce').fillna(0).astype(int)
    numeric_cols = [
        'test_return', 'test_max_dd', 'test_avg_pos', 'test_relative_return',
        'valid_return', 'valid_max_dd', 'selection_loss', 'relative_delta_vs_bc', 'max_dd_delta_vs_bc'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

recovery_metrics = load_candidate_metrics(RECOVERY_REPLAY, 'recovery_participation')
bull_metrics = load_candidate_metrics(BULL_REPLAY, 'bull_reentry_prior')
frames = [recovery_metrics, bull_metrics]
if (CAP10_REPLAY / 'checkpoint_candidate_metrics.csv').exists():
    frames.append(load_candidate_metrics(CAP10_REPLAY, 'cap10_reference'))
all_metrics = pd.concat(frames, ignore_index=True)

join_cols = ['fold', 'candidate_kind', 'iter']
comparison = recovery_metrics.merge(
    bull_metrics,
    on=join_cols,
    suffixes=('_recovery', '_bull'),
    how='outer',
)
comparison['test_return_delta_pp'] = comparison['test_return_recovery'] - comparison['test_return_bull']
comparison['relative_return_delta_pp'] = comparison['test_relative_return_recovery'] - comparison['test_relative_return_bull']
comparison['max_dd_delta_pp'] = comparison['test_max_dd_recovery'] - comparison['test_max_dd_bull']
comparison['avg_pos_delta'] = comparison['test_avg_pos_recovery'] - comparison['test_avg_pos_bull']
comparison['selection_loss_delta'] = comparison['selection_loss_recovery'] - comparison['selection_loss_bull']
comparison = comparison.sort_values(['fold', 'candidate_kind', 'iter']).reset_index(drop=True)

comparison_path = CARD / 'candidate_delta_vs_prior.csv'
comparison.to_csv(comparison_path, index=False)

selection_frames = []
for path, label in [(RECOVERY_REPLAY, 'recovery_participation'), (BULL_REPLAY, 'bull_reentry_prior')]:
    selection = pd.read_csv(path / 'checkpoint_selection_by_fold.csv')
    selection['source'] = label
    selection_frames.append(selection)
selection_by_fold = pd.concat(selection_frames, ignore_index=True)
selection_by_fold.to_csv(CARD / 'selection_by_fold_sources.csv', index=False)

wf2020_compact = comparison.loc[
    comparison['fold'].eq('wf_2020'),
    [
        'fold', 'candidate_kind', 'iter',
        'test_return_recovery', 'test_return_bull', 'test_return_delta_pp',
        'test_relative_return_recovery', 'test_relative_return_bull', 'relative_return_delta_pp',
        'test_max_dd_recovery', 'test_max_dd_bull', 'max_dd_delta_pp',
        'test_avg_pos_recovery', 'test_avg_pos_bull', 'avg_pos_delta',
    ],
]
print(wf2020_compact.to_string(index=False))


   fold    candidate_kind  iter  test_return_recovery  test_return_bull  test_return_delta_pp  test_relative_return_recovery  test_relative_return_bull  relative_return_delta_pp  test_max_dd_recovery  test_max_dd_bull  max_dd_delta_pp  test_avg_pos_recovery  test_avg_pos_bull  avg_pos_delta
wf_2020 bc_initialization     0             28.136997         55.597960            -27.460963                    -305.196351                -277.735388                -27.460963             14.863234         15.489488        -0.626254               0.378538           0.406077      -0.027539
wf_2020      ppo_retained     0             28.136997         55.597960            -27.460963                    -305.196351                -277.735388                -27.460963             14.863234         15.489488        -0.626254               0.378538           0.406077      -0.027539
wf_2020      ppo_retained     4             35.837155         50.224828            -14.387673                    -297.496193

## Target Semantics Audit

Current state-objective BC label construction calls `state_objective_target_position_from_row`:
- downtrend rows choose `risk_target_position`;
- rebound rows choose `rebound_target_position`;
- all other rows choose `teacher_position`.

This audit checks whether the recovery-specific column `recovery_participation_target_position` is aligned with that consumed target. Rows where the recovery column differs from the consumed target are rows where the new target semantics are not directly represented in the BC label.


In [3]:
def bool_series(df: pd.DataFrame, col: str) -> pd.Series:
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    s = df[col]
    if s.dtype == bool:
        return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float).ne(0.0)
    return s.fillna('').astype(str).str.strip().str.lower().isin({'1', 'true', 'yes', 'y'})

contract = pd.read_csv(RECOVERY_TARGET, parse_dates=['datetime', 'train_start', 'train_end', 'eval_start', 'eval_end'])
for col in [
    'teacher_position', 'rebound_target_position', 'risk_target_position',
    'recovery_participation_target_position', 'target_delta_vs_source',
    'source_risk_target_position', 'source_rebound_target_position', 'objective_weight',
]:
    if col in contract.columns:
        contract[col] = pd.to_numeric(contract[col], errors='coerce')

contract['consumed_state_objective_target_position'] = [
    state_objective_target_position_from_row(row)
    for row in contract.to_dict('records')
]
contract['target_diff_vs_consumed'] = (
    contract['recovery_participation_target_position']
    - contract['consumed_state_objective_target_position']
)
contract['target_diff_vs_consumed_abs'] = contract['target_diff_vs_consumed'].abs()
contract['ignored_recovery_target'] = contract['target_diff_vs_consumed_abs'].gt(1e-9)
contract['risk_changed_vs_source'] = (
    contract['risk_target_position'] - contract['source_risk_target_position']
).abs().gt(1e-9)
contract['rebound_changed_vs_source'] = (
    contract['rebound_target_position'] - contract['source_rebound_target_position']
).abs().gt(1e-9)
contract['target_delta_nonzero'] = contract['target_delta_vs_source'].abs().gt(1e-9)
contract['is_downtrend_bool'] = bool_series(contract, 'is_downtrend_state')
contract['is_rebound_bool'] = bool_series(contract, 'is_rebound_cluster')
contract['recovery_state_bool'] = bool_series(contract, 'recovery_participation_state_v2')
contract['risk_control_bool'] = bool_series(contract, 'risk_control_state_v2')
contract['conflict_bool'] = bool_series(contract, 'drawdown_recovery_conflict_state_v2')
contract['fallback_teacher_selected'] = ~contract['is_downtrend_bool'] & ~contract['is_rebound_bool']

train_mask = contract['datetime'].between(contract['train_start'], contract['train_end'])
train_contract = contract.loc[train_mask].copy()

rows = []
for fold, fold_df in train_contract.groupby('fold', sort=True):
    row = {
        'fold': fold,
        'train_rows': int(len(fold_df)),
        'target_delta_nonzero_rows': int(fold_df['target_delta_nonzero'].sum()),
        'risk_changed_rows': int(fold_df['risk_changed_vs_source'].sum()),
        'rebound_changed_rows': int(fold_df['rebound_changed_vs_source'].sum()),
        'ignored_recovery_target_rows': int(fold_df['ignored_recovery_target'].sum()),
        'ignored_recovery_target_share': float(fold_df['ignored_recovery_target'].mean()) if len(fold_df) else 0.0,
        'downtrend_rows': int(fold_df['is_downtrend_bool'].sum()),
        'rebound_rows': int(fold_df['is_rebound_bool'].sum()),
        'fallback_teacher_rows': int(fold_df['fallback_teacher_selected'].sum()),
        'recovery_state_rows': int(fold_df['recovery_state_bool'].sum()),
        'risk_control_rows': int(fold_df['risk_control_bool'].sum()),
        'conflict_rows': int(fold_df['conflict_bool'].sum()),
        'ignored_inside_recovery_state_rows': int((fold_df['ignored_recovery_target'] & fold_df['recovery_state_bool']).sum()),
        'ignored_inside_risk_control_rows': int((fold_df['ignored_recovery_target'] & fold_df['risk_control_bool']).sum()),
        'ignored_inside_conflict_rows': int((fold_df['ignored_recovery_target'] & fold_df['conflict_bool']).sum()),
        'recovery_col_mean': float(fold_df['recovery_participation_target_position'].mean()) if len(fold_df) else math.nan,
        'consumed_target_mean': float(fold_df['consumed_state_objective_target_position'].mean()) if len(fold_df) else math.nan,
        'risk_target_mean': float(fold_df['risk_target_position'].mean()) if len(fold_df) else math.nan,
        'rebound_target_mean': float(fold_df['rebound_target_position'].mean()) if len(fold_df) else math.nan,
        'teacher_position_mean': float(fold_df['teacher_position'].mean()) if len(fold_df) else math.nan,
        'mean_diff_recovery_minus_consumed': float(fold_df['target_diff_vs_consumed'].mean()) if len(fold_df) else math.nan,
        'mean_abs_diff_recovery_minus_consumed': float(fold_df['target_diff_vs_consumed_abs'].mean()) if len(fold_df) else math.nan,
    }
    rows.append(row)

audit = pd.DataFrame(rows)
audit_path = CARD / 'target_semantics_audit.csv'
audit.to_csv(audit_path, index=False)

ignored_cols = [
    'datetime', 'fold', 'candidate', 'teacher_position', 'risk_target_position', 'rebound_target_position',
    'recovery_participation_target_position', 'consumed_state_objective_target_position',
    'target_diff_vs_consumed', 'target_delta_vs_source', 'is_downtrend_state', 'is_rebound_cluster',
    'risk_control_state_v2', 'recovery_participation_state_v2', 'drawdown_recovery_conflict_state_v2',
    'target_recipe_id', 'target_reason',
]
ignored_rows = train_contract.loc[train_contract['ignored_recovery_target'], ignored_cols].copy()
ignored_rows.to_csv(CARD / 'ignored_recovery_target_rows.csv', index=False)

print(audit.to_string(index=False))
print('ignored rows written', len(ignored_rows))


              fold  train_rows  target_delta_nonzero_rows  risk_changed_rows  rebound_changed_rows  ignored_recovery_target_rows  ignored_recovery_target_share  downtrend_rows  rebound_rows  fallback_teacher_rows  recovery_state_rows  risk_control_rows  conflict_rows  ignored_inside_recovery_state_rows  ignored_inside_risk_control_rows  ignored_inside_conflict_rows  recovery_col_mean  consumed_target_mean  risk_target_mean  rebound_target_mean  teacher_position_mean  mean_diff_recovery_minus_consumed  mean_abs_diff_recovery_minus_consumed
     wf_2020_train        2470                        949                949                   156                           226                       0.091498             781           149                   1570                  226                839             54                                 164                                71                             9           0.304600              0.297078          0.304600             0.449741        

## Decision Artifacts

The decision is intentionally diagnostic: no model is promoted by this notebook. It records whether the current recovery target is a valid next training surface or whether the target semantics must be rebuilt before another checkpoint replay.


In [4]:
def finite_or_none(value):
    if value is None:
        return None
    try:
        value = float(value)
    except (TypeError, ValueError):
        return None
    if not math.isfinite(value):
        return None
    return value

wf2020_bc = comparison.loc[
    comparison['fold'].eq('wf_2020')
    & comparison['candidate_kind'].eq('bc_initialization')
    & comparison['iter'].eq(0)
].iloc[0]
wf2020_iter4 = comparison.loc[
    comparison['fold'].eq('wf_2020')
    & comparison['candidate_kind'].eq('ppo_retained')
    & comparison['iter'].eq(4)
]
iter4_row = wf2020_iter4.iloc[0] if len(wf2020_iter4) else None
wf2020_audit = audit.loc[audit['fold'].eq('wf_2020_train')]
wf2020_audit_row = wf2020_audit.iloc[0].to_dict() if len(wf2020_audit) else {}

strict_selected = pd.read_csv(RECOVERY_REPLAY / 'checkpoint_selection_by_fold.csv')
strict_pass_count = int(strict_selected['selected_candidate_gate'].astype(bool).sum())
strict_fold_count = int(len(strict_selected))
all_strict_pass = bool(strict_pass_count == strict_fold_count)

summary = {
    'experiment_id': 'wf2020_recovery_checkpoint_failure_attribution_v1',
    'decision': 'wf2020_recovery_target_semantics_mismatch_blocks_checkpoint_replay_no_promotion',
    'promote_actor': False,
    'strict_selected_fold_pass_count': strict_pass_count,
    'strict_selected_fold_count': strict_fold_count,
    'all_strict_folds_pass': all_strict_pass,
    'wf2020_bc_return_recovery': finite_or_none(wf2020_bc['test_return_recovery']),
    'wf2020_bc_return_prior_bull': finite_or_none(wf2020_bc['test_return_bull']),
    'wf2020_bc_return_delta_pp': finite_or_none(wf2020_bc['test_return_delta_pp']),
    'wf2020_bc_relative_return_delta_pp': finite_or_none(wf2020_bc['relative_return_delta_pp']),
    'wf2020_bc_max_dd_delta_pp': finite_or_none(wf2020_bc['max_dd_delta_pp']),
    'wf2020_bc_avg_pos_delta': finite_or_none(wf2020_bc['avg_pos_delta']),
    'wf2020_iter4_return_delta_pp': finite_or_none(iter4_row['test_return_delta_pp']) if iter4_row is not None else None,
    'wf2020_iter4_relative_return_delta_pp': finite_or_none(iter4_row['relative_return_delta_pp']) if iter4_row is not None else None,
    'wf2020_iter4_max_dd_delta_pp': finite_or_none(iter4_row['max_dd_delta_pp']) if iter4_row is not None else None,
    'wf2020_train_rows': int(wf2020_audit_row.get('train_rows', 0)),
    'wf2020_target_delta_nonzero_rows': int(wf2020_audit_row.get('target_delta_nonzero_rows', 0)),
    'wf2020_ignored_recovery_target_rows': int(wf2020_audit_row.get('ignored_recovery_target_rows', 0)),
    'wf2020_ignored_recovery_target_share': finite_or_none(wf2020_audit_row.get('ignored_recovery_target_share')),
    'root_cause': 'recovery_participation_target_position is a diagnostic/new target column but the current state_objective BC target chooser only consumes downtrend risk, rebound, or teacher targets.',
    'next_action': 'project recovery semantics into consumed target columns or add a tested explicit target-priority interface before another three-fold checkpoint replay.',
    'artifacts': {
        'candidate_delta_vs_prior': str((CARD / 'candidate_delta_vs_prior.csv').relative_to(ROOT)),
        'target_semantics_audit': str((CARD / 'target_semantics_audit.csv').relative_to(ROOT)),
        'ignored_recovery_target_rows': str((CARD / 'ignored_recovery_target_rows.csv').relative_to(ROOT)),
        'selection_by_fold_sources': str((CARD / 'selection_by_fold_sources.csv').relative_to(ROOT)),
    },
}

selection = {
    'experiment_id': summary['experiment_id'],
    'selected_for_promotion': None,
    'decision': summary['decision'],
    'reason': summary['root_cause'],
    'strict_gate': {
        'selected_fold_pass_count': strict_pass_count,
        'selected_fold_count': strict_fold_count,
        'all_selected_folds_pass': all_strict_pass,
    },
    'next_action': summary['next_action'],
}

(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n')
(CARD / 'selection.json').write_text(json.dumps(selection, indent=2, sort_keys=True) + '\n')

conclusion = f"""# WF2020 Recovery Checkpoint Failure Attribution v1

Decision: `{summary['decision']}`. No actor is promoted.

What ran:
- compared latest recovery checkpoint replay against prior bull-reentry checkpoint replay;
- audited the recovery target contract against the current `state_objective_target_position_from_row` target chooser;
- refreshed sidecar evidence in this card.

Key metrics:
- Strict selected replay pass count: {strict_pass_count}/{strict_fold_count} folds.
- `wf_2020` BC return changed from {summary['wf2020_bc_return_prior_bull']:.6f}% to {summary['wf2020_bc_return_recovery']:.6f}% ({summary['wf2020_bc_return_delta_pp']:.6f} pp).
- `wf_2020` BC relative return delta vs prior: {summary['wf2020_bc_relative_return_delta_pp']:.6f} pp.
- `wf_2020` BC MaxDD delta vs prior: {summary['wf2020_bc_max_dd_delta_pp']:.6f} pp.
- `wf_2020_train` changed-target rows: {summary['wf2020_target_delta_nonzero_rows']} of {summary['wf2020_train_rows']}.
- `wf_2020_train` rows where recovery target differs from consumed BC target: {summary['wf2020_ignored_recovery_target_rows']} ({summary['wf2020_ignored_recovery_target_share']:.6f}).

Root cause:
`recovery_participation_target_position` is not a consumed target surface for current state-objective BC labels. The current target chooser selects `risk_target_position` for downtrend rows, `rebound_target_position` for rebound rows, and `teacher_position` otherwise. Any recovery semantics that are only represented in the new recovery column are therefore diagnostic, not training behavior.

Risk notes:
- Continuing checkpoint replay on this contract would test a partially mismatched label surface.
- A shared BC target-priority change is higher blast radius and needs unit tests before use.

Next action:
Project recovery semantics into the existing consumed columns/state flags, or add a tested explicit target-priority interface, then rerun the three-fold checkpoint metric replay.

Artifacts:
- `candidate_delta_vs_prior.csv`
- `target_semantics_audit.csv`
- `ignored_recovery_target_rows.csv`
- `selection_by_fold_sources.csv`
- `summary.json`
- `selection.json`
"""
(CARD / 'conclusion.md').write_text(conclusion)

print(json.dumps(summary, indent=2, sort_keys=True))


{
  "all_strict_folds_pass": false,
  "artifacts": {
    "candidate_delta_vs_prior": "runtime/model_selection/card/wf2020_recovery_checkpoint_failure_attribution_v1/candidate_delta_vs_prior.csv",
    "ignored_recovery_target_rows": "runtime/model_selection/card/wf2020_recovery_checkpoint_failure_attribution_v1/ignored_recovery_target_rows.csv",
    "selection_by_fold_sources": "runtime/model_selection/card/wf2020_recovery_checkpoint_failure_attribution_v1/selection_by_fold_sources.csv",
    "target_semantics_audit": "runtime/model_selection/card/wf2020_recovery_checkpoint_failure_attribution_v1/target_semantics_audit.csv"
  },
  "decision": "wf2020_recovery_target_semantics_mismatch_blocks_checkpoint_replay_no_promotion",
  "experiment_id": "wf2020_recovery_checkpoint_failure_attribution_v1",
  "next_action": "project recovery semantics into consumed target columns or add a tested explicit target-priority interface before another three-fold checkpoint replay.",
  "promote_actor": false

## Conclusion

The recovery replay is not promotable. Its `wf_2020` return regressed versus the prior bull-reentry replay, and the recovery-specific target column is not a first-class consumed target in the current state-objective BC label builder. The next implementation step should be a tested target semantics repair before another training or checkpoint replay loop.
